# UD5.07. Aprendizaje por transferencia

**Módulo 5073 · Programación de Inteligencia Artificial · Curso 2026/27**
Bloque 15 de los apuntes · Criterio **2.d**

---

Una red entrenada sobre ImageNet —1,2 millones de imágenes y mil clases— ha aprendido en
sus primeras capas a detectar bordes, texturas, círculos y gradientes de color. Esas cosas
**no dependen del problema**: sirven igual para clasificar radiografías que para
clasificar flores. Lo específico del problema está en las últimas capas.

Así que se reaprovecha lo genérico y se sustituye lo específico. La consecuencia práctica
es grande: **con unos cientos de imágenes por clase se llega donde desde cero no se
llegaría con decenas de miles**.

> **Con imágenes de tamaño real, entrenar desde cero es casi siempre la decisión
> equivocada.** Este cuaderno lo mide, en lugar de afirmarlo.

### Aviso de coste, y la guarda

En la máquina del aula no hay GPU. Medido con MobileNetV2, la **extracción de rasgos**
cuesta entre 0,008 y 0,02 segundos por imagen y el **ajuste fino**, entre 0,04 y 0,11: un
factor cinco. Dos mil imágenes y cinco épocas de ajuste fino son unos diez minutos por
ejecución, que es tolerable para comprobarlo una vez e inservible para iterar.

Por eso el ajuste fino va detrás de una bandera, con el mismo patrón que las claves de
Azure de la UD2: si no hay acelerador, el cuaderno avisa y usa la versión reducida en lugar
de quedarse colgado.

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras
import tensorflow as tf

SEMILLA = 20262027
keras.utils.set_random_seed(SEMILLA)

HAY_GPU = len(tf.config.list_physical_devices("GPU")) > 0

# --- La bandera. Ponla a True en Colab con GPU. ---
AJUSTE_FINO = HAY_GPU

# Tamaños de trabajo. Con GPU se pueden subir los tres.
N_ENTRENA = 2000 if HAY_GPU else 1000
N_PRUEBA = 1000
TAMANO = 96                 # MobileNetV2 admite 96, 128, 160, 192 y 224

print(f"GPU disponible: {HAY_GPU}")
print(f"Ajuste fino:    {AJUSTE_FINO}")
print(f"Trabajando con {N_ENTRENA} imagenes de entrenamiento a {TAMANO}x{TAMANO}.")
if not HAY_GPU:
    print()
    print("Sin GPU: la extraccion de rasgos si cabe en el aula (unos 20 s por cada")
    print("mil imagenes); el ajuste fino se salta y se explica con las cifras medidas.")

---

## 1. Los datos: CIFAR-10

Diez clases de objetos cotidianos, 32×32 en color. Es pequeño, se descarga desde Keras, y
sobre todo **es del mismo tipo que ImageNet**: fotografías en color de objetos naturales.
Esa es la condición que hace que la transferencia funcione, y conviene verla explícita.

In [ ]:
CLASES = ["avion", "automovil", "pajaro", "gato", "ciervo",
          "perro", "rana", "caballo", "barco", "camion"]

(X_todo, y_todo), (X_p, y_p) = keras.datasets.cifar10.load_data()
y_todo, y_p = y_todo.ravel(), y_p.ravel()

X_ent = X_todo[:N_ENTRENA]
y_ent = y_todo[:N_ENTRENA]
X_val = X_todo[N_ENTRENA:N_ENTRENA + 500]
y_val = y_todo[N_ENTRENA:N_ENTRENA + 500]
X_pru = X_p[:N_PRUEBA]
y_pru = y_p[:N_PRUEBA]

print(f"entrenamiento {X_ent.shape}   validacion {X_val.shape}   prueba {X_pru.shape}")
print(f"rango de los pixeles: [{X_ent.min()}, {X_ent.max()}]  dtype {X_ent.dtype}")
print()
print("Reparto de clases en el entrenamiento:")
print(pd.Series(y_ent).map(dict(enumerate(CLASES))).value_counts().to_string())

In [ ]:
fig, ejes = plt.subplots(2, 5, figsize=(11, 4.6))
rng = np.random.default_rng(SEMILLA)
for eje, idx in zip(ejes.ravel(), rng.choice(len(X_ent), 10, replace=False)):
    eje.imshow(X_ent[idx])
    eje.set_title(CLASES[y_ent[idx]], fontsize=9)
    eje.axis("off")
fig.suptitle("CIFAR-10: fotografias en color, como las de ImageNet. "
             "Esa semejanza es lo que hace que la transferencia funcione", y=1.02)
fig.tight_layout()
plt.show()

---

## 2. El punto de referencia: una CNN desde cero

Sin esto no se puede decir cuánto aporta la transferencia. Es la misma regla de siempre.

In [ ]:
def cnn_desde_cero():
    return keras.Sequential([
        keras.layers.Input(shape=(32, 32, 3)),
        keras.layers.Rescaling(1.0 / 255),
        keras.layers.Conv2D(32, 3, activation="relu", padding="same"),
        keras.layers.MaxPooling2D(),
        keras.layers.Conv2D(64, 3, activation="relu", padding="same"),
        keras.layers.MaxPooling2D(),
        keras.layers.Conv2D(128, 3, activation="relu", padding="same"),
        keras.layers.GlobalAveragePooling2D(),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(10, activation="softmax"),
    ])


keras.utils.set_random_seed(SEMILLA)
propia = cnn_desde_cero()
propia.compile(optimizer=keras.optimizers.Adam(1e-3),
               loss="sparse_categorical_crossentropy", metrics=["accuracy"])

t0 = time.perf_counter()
h_propia = propia.fit(X_ent, y_ent, epochs=20, batch_size=32,
                      validation_data=(X_val, y_val), verbose=0,
                      callbacks=[keras.callbacks.EarlyStopping(
                          monitor="val_accuracy", mode="max", patience=6,
                          restore_best_weights=True)])
t_propia = time.perf_counter() - t0
_, acc_propia = propia.evaluate(X_pru, y_pru, verbose=0)

print(f"CNN desde cero: {propia.count_params():,} parametros, "
      f"{len(h_propia.history['loss'])} epocas, {t_propia:.1f} s")
print(f"exactitud de prueba: {acc_propia:.4f}")
print(f"exactitud de la clase mayoritaria: {pd.Series(y_pru).value_counts(normalize=True).max():.4f}")

---

## 3. Extracción de rasgos

La base congelada, y una cabeza nueva encima. Tres detalles que hay que hacer bien:

1. **Redimensionar** a un tamaño que la base admita: MobileNetV2 acepta 96, 128, 160, 192 y
   224.
2. **El preprocesamiento que espera la base.** MobileNetV2 quiere $[-1, 1]$; ResNet50,
   canales centrados en la media de ImageNet y en orden BGR; EfficientNet, $[0, 255]$ sin
   tocar. Normalizar a $[0,1]$ "porque es lo que se hace" produce un modelo que funciona
   **algo peor sin dar ningún error**, que es la peor forma de fallar.
3. **`base.trainable = False`** antes de compilar.

In [ ]:
def base_mobilenet():
    base = keras.applications.MobileNetV2(
        input_shape=(TAMANO, TAMANO, 3),
        include_top=False,        # sin la cabeza de las 1000 clases de ImageNet
        weights="imagenet",       # con los pesos ya aprendidos
    )
    base.trainable = False        # congelada
    return base


def modelo_transferido(base, aumento=True):
    capas = [keras.layers.Input(shape=(32, 32, 3)),
             keras.layers.Resizing(TAMANO, TAMANO)]
    if aumento:
        capas += [keras.layers.RandomFlip("horizontal"),
                  keras.layers.RandomRotation(0.05),
                  keras.layers.RandomZoom(0.1)]
    capas += [
        # El preprocesamiento de MobileNetV2: de [0, 255] a [-1, 1].
        keras.layers.Rescaling(1.0 / 127.5, offset=-1.0),
        base,
        keras.layers.GlobalAveragePooling2D(),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(10, activation="softmax"),
    ]
    return keras.Sequential(capas)


keras.utils.set_random_seed(SEMILLA)
base = base_mobilenet()
transferido = modelo_transferido(base)
transferido.compile(optimizer=keras.optimizers.Adam(1e-3),
                    loss="sparse_categorical_crossentropy", metrics=["accuracy"])

entrenables = sum(np.prod(v.shape) for v in transferido.trainable_variables)
print(f"parametros totales:     {transferido.count_params():,}")
print(f"parametros ENTRENABLES: {int(entrenables):,}")
print()
print("Casi todo el modelo esta congelado: lo unico que se entrena es la cabeza.")
print("Por eso cuesta tan poco, y por eso funciona con tan pocas imagenes.")

In [ ]:
t0 = time.perf_counter()
h_trans = transferido.fit(X_ent, y_ent, epochs=12, batch_size=32,
                          validation_data=(X_val, y_val), verbose=0,
                          callbacks=[keras.callbacks.EarlyStopping(
                              monitor="val_accuracy", mode="max", patience=4,
                              restore_best_weights=True)])
t_trans = time.perf_counter() - t0
_, acc_trans = transferido.evaluate(X_pru, y_pru, verbose=0)

print(f"extraccion de rasgos: {len(h_trans.history['loss'])} epocas, {t_trans:.1f} s")
print(f"  = {t_trans / len(h_trans.history['loss']) / N_ENTRENA * 1000:.1f} ms por imagen y epoca")
print(f"exactitud de prueba: {acc_trans:.4f}")

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 4.2))
for eje, clave, etiqueta in zip(ejes, ["loss", "accuracy"],
                                ["entropia cruzada", "exactitud"]):
    for nombre, h in [("CNN desde cero", h_propia), ("transferencia", h_trans)]:
        linea, = eje.plot(h.history[clave], label=f"{nombre} (entrena)")
        eje.plot(h.history["val_" + clave], "--", color=linea.get_color(),
                 label=f"{nombre} (valida)")
    eje.set_xlabel("epoca");  eje.set_ylabel(etiqueta)
    eje.legend(fontsize=7)
ejes[0].set_title("La transferencia arranca ya sabiendo mirar")
ejes[1].set_title("Y llega mas alto, con menos epocas")
fig.tight_layout()
plt.show()

### La comparación

In [ ]:
comparacion = pd.DataFrame([
    {"modelo": "clase mayoritaria", "parametros entrenables": 0,
     "exactitud prueba": float(pd.Series(y_pru).value_counts(normalize=True).max()),
     "segundos": 0.0},
    {"modelo": "CNN desde cero", "parametros entrenables": propia.count_params(),
     "exactitud prueba": acc_propia, "segundos": t_propia},
    {"modelo": "MobileNetV2, base congelada", "parametros entrenables": int(entrenables),
     "exactitud prueba": acc_trans, "segundos": t_trans},
])
print(comparacion.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print()
print(f"Mejora de la transferencia sobre la CNN propia: "
      f"{(acc_trans - acc_propia) * 100:+.1f} puntos de exactitud")
print(f"con {int(entrenables) / propia.count_params():.2f} veces sus parametros entrenables.")

> **Y esta es la comparación que cierra el criterio 2.c de toda la unidad.** En el cuaderno
> `UD5_04`, con datos tabulares y características construidas a mano, la red neuronal
> perdió contra una regla de una línea. Aquí, con imágenes, gana por una diferencia que no
> admite discusión.
>
> Es la misma pregunta y el mismo método. Lo que cambia es si hay representación que
> aprender, y en una imagen la hay toda.

---

## 4. El preprocesamiento equivocado

El error que no da error. Vamos a medirlo: la misma base, la misma cabeza, la misma
semilla, y como única diferencia un `Rescaling(1/255)` en lugar del $[-1,1]$ que
MobileNetV2 espera.

In [ ]:
keras.utils.set_random_seed(SEMILLA)
base_mal = base_mobilenet()
mal = keras.Sequential([
    keras.layers.Input(shape=(32, 32, 3)),
    keras.layers.Resizing(TAMANO, TAMANO),
    keras.layers.RandomFlip("horizontal"),
    keras.layers.RandomRotation(0.05),
    keras.layers.RandomZoom(0.1),
    keras.layers.Rescaling(1.0 / 255),        # <-- [0, 1] en vez de [-1, 1]
    base_mal,
    keras.layers.GlobalAveragePooling2D(),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(10, activation="softmax"),
])
mal.compile(optimizer=keras.optimizers.Adam(1e-3),
            loss="sparse_categorical_crossentropy", metrics=["accuracy"])
mal.fit(X_ent, y_ent, epochs=12, batch_size=32, validation_data=(X_val, y_val),
        verbose=0, callbacks=[keras.callbacks.EarlyStopping(
            monitor="val_accuracy", mode="max", patience=4, restore_best_weights=True)])
_, acc_mal = mal.evaluate(X_pru, y_pru, verbose=0)

print(f"preprocesamiento correcto  [-1, 1]: {acc_trans:.4f}")
print(f"preprocesamiento generico  [ 0, 1]: {acc_mal:.4f}")
print(f"diferencia: {acc_trans - acc_mal:+.4f}")
print()
print("Ni una excepcion, ni un aviso. El modelo entrena, converge, y da una")
print("cifra que parece razonable. Solo sabes que esta mal si tienes el otro")
print("al lado. Por eso se usa la funcion que trae cada modelo:")
print("  from keras.applications.mobilenet_v2 import preprocess_input")

---

## 5. Ajuste fino

Descongelar las últimas capas de la base y seguir entrenando **con una tasa de aprendizaje
diez o cien veces menor**. Tres reglas, y saltarse cualquiera de las tres estropea el
resultado:

1. **Nunca antes de que la cabeza converja.** El gradiente enorme de una cabeza recién
   inicializada destroza en un solo paso unos pesos que costaron semanas de cómputo. Se
   llama *olvido catastrófico*.
2. **Tasa entre `1e-5` y `1e-4`**, nunca la normal.
3. **Hay que recompilar.** Cambiar `base.trainable` después de compilar no tiene efecto
   hasta que se vuelve a llamar a `compile`.

In [ ]:
def prepara_ajuste_fino(modelo, base, capas_a_descongelar=30, tasa=1e-5):
    base.trainable = True
    # Solo las ultimas capas: las primeras detectan bordes, y eso no hay que
    # reaprenderlo. Las capas de normalizacion por lotes se dejan congeladas
    # porque su estadistica acumulada se estropea con lotes pequeños.
    for capa in base.layers[:-capas_a_descongelar]:
        capa.trainable = False
    for capa in base.layers:
        if isinstance(capa, keras.layers.BatchNormalization):
            capa.trainable = False

    # RECOMPILAR. Sin esto, el cambio de trainable no tiene efecto.
    modelo.compile(optimizer=keras.optimizers.Adam(tasa),
                   loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return modelo


if AJUSTE_FINO:
    transferido = prepara_ajuste_fino(transferido, base)
    entrenables_fino = sum(np.prod(v.shape) for v in transferido.trainable_variables)
    print(f"parametros entrenables ahora: {int(entrenables_fino):,} "
          f"(antes {int(entrenables):,})")

    t0 = time.perf_counter()
    h_fino = transferido.fit(X_ent, y_ent, epochs=5, batch_size=32,
                             validation_data=(X_val, y_val), verbose=0)
    t_fino = time.perf_counter() - t0
    _, acc_fino = transferido.evaluate(X_pru, y_pru, verbose=0)
    print(f"ajuste fino: 5 epocas, {t_fino:.1f} s ({t_fino / 5:.1f} s/epoca)")
    print(f"exactitud: {acc_trans:.4f} -> {acc_fino:.4f} ({acc_fino - acc_trans:+.4f})")
else:
    print("AJUSTE_FINO esta a False: no hay GPU y esta parte costaria minutos.")
    print()
    print("Las cifras medidas en la maquina del modulo, con MobileNetV2:")
    print()
    print(f"  {'tamaño':>8} {'imagenes':>9} {'extraccion':>12} {'1 epoca de ajuste fino':>24}")
    print("  " + "-" * 56)
    for tamano, n, extraccion, fino in [("96x96", 1000, "8,4 s", "42 s"),
                                        ("160x160", 1000, "19,8 s", "58 s"),
                                        ("224x224", 500, "7,0 s", "55 s")]:
        print(f"  {tamano:>8} {n:>9} {extraccion:>12} {fino:>24}")
    print()
    print("  Normalizado: extraccion entre 0,008 y 0,02 s por imagen;")
    print("  ajuste fino entre 0,04 y 0,11. Un factor CINCO.")
    print()
    print("En Colab con GPU, pon AJUSTE_FINO = True al principio del cuaderno.")

### Por qué la normalización por lotes se queda congelada

Es el detalle que más resultados estropea y casi nunca se explica. Una capa
`BatchNormalization` guarda una media y una varianza acumuladas de todo el entrenamiento de
ImageNet. Si se descongela, esas estadísticas empiezan a actualizarse con los lotes
pequeños del conjunto nuevo, que son ruidosos, y la base deja de funcionar como funcionaba.

La regla práctica: **al hacer ajuste fino, las capas de normalización por lotes de la base
se dejan en modo de inferencia.**

---

## 6. Qué ha aprendido la base

Los rasgos que MobileNetV2 extrae son un vector de 1.280 números por imagen. Si esos
números separan bien las clases, la cabeza lo tiene fácil, y eso se puede ver antes de
entrenar nada.

In [ ]:
# Extraer los rasgos de la base congelada, sin cabeza.
extractor = keras.Sequential([
    keras.layers.Input(shape=(32, 32, 3)),
    keras.layers.Resizing(TAMANO, TAMANO),
    keras.layers.Rescaling(1.0 / 127.5, offset=-1.0),
    base_mobilenet(),
    keras.layers.GlobalAveragePooling2D(),
])

t0 = time.perf_counter()
rasgos = extractor.predict(X_pru[:500], verbose=0)
print(f"{rasgos.shape}  en {time.perf_counter() - t0:.1f} s")
print(f"  = {(time.perf_counter() - t0) / 500 * 1000:.1f} ms por imagen")

In [ ]:
# Proyeccion a dos dimensiones con PCA, que es lineal y por tanto honesta:
# si las clases ya se separan aqui, es que los rasgos las separan de verdad.
from sklearn.decomposition import PCA

proyeccion = PCA(n_components=2, random_state=SEMILLA).fit_transform(rasgos)

fig, ejes = plt.subplots(1, 2, figsize=(12, 5))
for eje, (datos, titulo) in zip(ejes, [
        (proyeccion, "Rasgos de MobileNetV2 (1280 dimensiones -> 2)"),
        (PCA(n_components=2, random_state=SEMILLA).fit_transform(
            X_pru[:500].reshape(500, -1) / 255.0),
         "Pixeles crudos (3072 dimensiones -> 2)")]):
    dispersion = eje.scatter(datos[:, 0], datos[:, 1], c=y_pru[:500],
                             cmap="tab10", s=14, alpha=0.8)
    eje.set_title(titulo, fontsize=10)
    eje.set_xlabel("componente 1");  eje.set_ylabel("componente 2")
fig.colorbar(dispersion, ax=ejes, ticks=range(10), label="clase",
             fraction=0.03).ax.set_yticklabels(CLASES, fontsize=7)
fig.suptitle("La base preentrenada ya separa las clases ANTES de entrenar nada. "
             "Los pixeles crudos, no", y=1.02)
plt.show()

> **Eso es lo que se transfiere.** No es el modelo: es la representación. Y por eso la
> cabeza puede ser una sola capa densa y aun así funcionar.

---

## 7. Dónde se equivoca, y cuánto cuesta

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

pred = transferido.predict(X_pru, verbose=0).argmax(axis=1)
mc = confusion_matrix(y_pru, pred, normalize="true")

fig, eje = plt.subplots(figsize=(7.5, 6.5))
imagen_mc = eje.imshow(mc, cmap="Blues", vmin=0, vmax=1)
eje.set_xticks(range(10), CLASES, rotation=45, ha="right", fontsize=8)
eje.set_yticks(range(10), CLASES, fontsize=8)
for i in range(10):
    for j in range(10):
        if mc[i, j] >= 0.03:
            eje.text(j, i, f"{mc[i, j]:.2f}", ha="center", va="center", fontsize=7,
                     color="white" if mc[i, j] > 0.5 else "black")
eje.set_xlabel("lo que dice el modelo");  eje.set_ylabel("lo que es")
eje.set_title("Normalizada por fila")
fig.colorbar(imagen_mc, ax=eje, fraction=0.046)
fig.tight_layout()
plt.show()

print(classification_report(y_pru, pred, target_names=CLASES, digits=3))

In [ ]:
# El coste en disco y en latencia: lo que decide si el modelo cabe en un movil.
import tempfile

filas = []
for nombre, modelo in [("CNN desde cero", propia),
                       ("MobileNetV2 transferido", transferido)]:
    with tempfile.TemporaryDirectory() as carpeta:
        ruta = os.path.join(carpeta, "m.keras")
        modelo.save(ruta)
        tamano = os.path.getsize(ruta) / 1e6

    una = X_pru[:1]
    modelo.predict(una, verbose=0)                       # calentar
    tiempos = []
    for _ in range(20):
        t0 = time.perf_counter()
        modelo.predict(una, verbose=0)
        tiempos.append(time.perf_counter() - t0)

    t0 = time.perf_counter()
    modelo.predict(X_pru[:256], batch_size=256, verbose=0)
    por_muestra_lote = (time.perf_counter() - t0) / 256

    filas.append({"modelo": nombre, "parametros": modelo.count_params(),
                  "MB en disco": tamano,
                  "ms con lote de 1": np.median(tiempos) * 1000,
                  "ms por muestra en lote de 256": por_muestra_lote * 1000})

print(pd.DataFrame(filas).to_string(index=False, float_format=lambda v: f"{v:.2f}"))
print()
print("La ultima columna frente a la penultima es la misma idea que la")
print("vectorizacion de la UD3: el coste fijo por llamada se reparte entre")
print("las muestras del lote. Por eso los servicios reales agrupan peticiones.")

---

## Ejercicios

### Ejercicio 1. El tamaño de entrada

Repite la extracción de rasgos con `TAMANO` en 96, 128, 160 y 224. Dibuja exactitud de
prueba y segundos frente al tamaño. ¿Dónde está el punto en el que dejar de crecer? Ten en
cuenta que CIFAR-10 es de 32×32: estás **ampliando**, no añadiendo detalle. Escribe una
frase sobre qué significa eso.

### Ejercicio 2. Cuántas imágenes hacen falta

Entrena la cabeza con 100, 250, 500, 1.000 y 2.000 imágenes, y haz lo mismo con la CNN
desde cero. Dibuja las dos curvas de exactitud frente al número de imágenes. **¿Dónde se
cruzan?** Ese cruce es la respuesta a *cuándo compensa la transferencia*, y es el resultado
más útil del cuaderno.

### Ejercicio 3. Otra base

Repite con `EfficientNetB0` y con `ResNet50`. Para cada una, usa **su** `preprocess_input`.
Compara exactitud, parámetros, tamaño en disco y latencia. Construye la tabla con la que
elegirías una para un móvil y otra para un servidor.

### Ejercicio 4. Un dominio lejano

Haz lo mismo sobre **Fashion-MNIST**, que es en escala de grises y de ropa sobre fondo
liso: un dominio mucho más lejano de ImageNet que CIFAR-10. Hay que repetir el canal tres
veces para que la base lo acepte. ¿Sigue ganando la transferencia? Relaciona el resultado
con la afirmación de que lo que se transfiere son bordes y texturas.

### Ejercicio 5. El olvido catastrófico

En Colab con GPU, haz el ajuste fino **sin bajar la tasa de aprendizaje**: descongela y
sigue con `1e-3`. Dibuja la curva. Deberías ver la exactitud desplomarse en la primera
época. Explica qué ha pasado con las palabras del apartado 5.

### Ejercicio 6. Congelar la normalización por lotes, o no

También con GPU: haz el ajuste fino dejando las capas `BatchNormalization` entrenables y
compáralo con dejarlas congeladas. Documenta la diferencia, que suele ser de varios puntos,
y explícala.

---

## Lo que hay que llevarse de aquí

1. **Lo que se transfiere es la representación, no el modelo.** Se ve proyectando los
   rasgos: separan las clases antes de entrenar nada.
2. **Con imágenes de tamaño real, entrenar desde cero es casi siempre la decisión
   equivocada.**
3. **La transferencia funciona cuando el dominio se parece.** CIFAR-10 son fotos, como
   ImageNet; un conjunto muy distinto transfiere peor.
4. **El preprocesamiento tiene que ser el que espera la base**, y equivocarlo no da ningún
   error: solo un resultado algo peor.
5. **Extracción de rasgos primero, ajuste fino después.** Nunca al revés.
6. **Tasa de diez a cien veces menor en el ajuste fino, y recompilar.**
7. **Las capas de normalización por lotes de la base se quedan congeladas.**
8. **Un modelo pequeño y uno grande no se eligen por su exactitud**, sino por exactitud,
   tamaño y latencia a la vez.
9. **Y la comparación con `UD5_04` cierra la unidad**: la misma pregunta del criterio 2.c,
   el mismo método, y la respuesta contraria, porque aquí sí hay representación que
   aprender.